# Setup Environment

In [ ]:
%load_ext autoreload
%autoreload 1

%aimport highlighting_precision
%aimport gradcam_APs
%aimport gradcam
%aimport vanilla_vgg
%aimport dataHelper
%aimport external

import warnings
from torch.serialization import SourceChangeWarning
warnings.filterwarnings("ignore", category=SourceChangeWarning)

%matplotlib inline
!nvidia-smi

In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ext = external.ExternalValidationContext(
    environment = 'duke',
    external_validation_path=Path('/redacted/path/to/duke'),
    data_path=Path('/redacted/path/to/data'),
    models=['iaia-bl', 'protopnet',  'vgg16'],
    margin_name_labels = [
        (0, 'circ_prob', 'circumscribed'),
        (1, 'indst_prob', 'indistinct'),
        (2, 'spic_prob', 'spiculated')
    ],
    malignancy_name_labels = [
        (0, 'mal_neg_prob', 'not_malignant'),
        (1, 'mal_pos_prob', 'malignant')
    ],
    subgroupings  = [
        ('all', 'all'), 
        ('manufacturer', 'ge'), ('manufacturer', 'hologic'),
        ('age_3_bucket', '<50'), ('age_3_bucket','50-69'), ('age_3_bucket', '>70'),
        ('age_2_bucket', '<=57'), ('age_2_bucket','>57'),
        ('race', 'Black or African American'), ('race', 'Caucasian/White'), ('race', 'Other/Unknown')
    ]
)

ext

# Duke Validation

## Activation Precision
These just setup the run and get preliminary results (n<<100). See the Export section below for n=100.

### GradCAM for VGG16

In [ ]:
gradcam_aps = ext.run_gradcams(num_bootstraps=3)
gradcam_aps.pipe(display)
gradcam_aps.groupby(['method']).mean().pipe(display)

### ProtoPNets

In [ ]:
protopnets_aps = ext.classwise_activation_precision_for_protopnets(num_bootstraps=3)
protopnets_aps.pipe(display)
protopnets_aps.groupby(['model']).mean().pipe(display)

# Model Predictions
## Margin Predictions

In [ ]:
margin_predictions = ext.margin_predictions_for_models(image_has_rois=True)
margin_predictions.to_csv(ext.external_validation_path/f'{ext.environment}_margin_predictions.csv', index=False)
margin_predictions.groupby('model_name').count()

In [ ]:
__prob_sums = margin_predictions[['circ_prob', 'indst_prob', 'spic_prob']].sum(axis=1).round(2).value_counts()
assert len(__prob_sums) == 1 and 1.0 in __prob_sums
__prob_sums

### Analyze Results

In [ ]:
margin_predictions = pd.read_csv(ext.external_validation_path/f'{ext.environment}_margin_predictions.csv')
margin_predictions.groupby('model_name').apply(lambda group: group.sample(2))

In [ ]:
ext.plot_margin_probs(margin_predictions)

## Malignancy Prediction

### Add Malignancy Labels

In [ ]:
label_demographic_df = pd.read_csv(ext.external_validation_path/'combined_dataset_v2_age_race_machine.csv')
display(label_demographic_df.columns)
label_demographic_df.head(3)

In [ ]:
label_demographic_df.shape, margin_predictions.shape

In [ ]:
margin_pred_wlabels_df = margin_predictions.merge(label_demographic_df, how='left', left_on='patient_id', right_on='filename_no_ext')
margin_pred_wlabels_df['manufacturer_orig'] = margin_pred_wlabels_df['manufacturer']
margin_pred_wlabels_df.head(3)

In [ ]:
def clean_manufacturer(manufacturer):
    if manufacturer == 'GE MEDICAL SYSTEMS':
        return 'ge'
    elif manufacturer == 'HOLOGIC, Inc.':
        return 'hologic'
    else:
        raise ValueError('Invalid manufacturer')

margin_pred_wlabels_df['race'] = margin_pred_wlabels_df['first race']

margin_pred_wlabels_df['age_3_bucket'] = margin_pred_wlabels_df['age'].apply(external.age_3_bucket)
margin_pred_wlabels_df['age_2_bucket'] = margin_pred_wlabels_df['age'].apply(external.age_2_bucket)

margin_pred_wlabels_df['manufacturer'] = margin_pred_wlabels_df['manufacturer_orig'].apply(clean_manufacturer)

margin_pred_wlabels_df[['first race', 'race', 'age', 'age_2_bucket', 'age_3_bucket', 'manufacturer_orig', 'manufacturer']].head(3)

In [ ]:
label_matches = (margin_pred_wlabels_df['margin'] == margin_pred_wlabels_df['lesion margin'].str.lower())

margin_pred_wlabels_df[~label_matches][['margin', 'lesion margin', 'margin_label']].head(3)

In [ ]:
margin_pred_wlabels_df['pathology'].unique()

In [ ]:
score_label_df = margin_pred_wlabels_df.copy()
score_label_df['mass_uid'] = score_label_df['patient_id'].str.split('_').str[2]
score_label_df['image_id'] = score_label_df['patient_id']
score_label_df['patient_id'] = score_label_df['fake id']

def race_simple(race):

    if race == 'Not Reported/Declined' or race == 'Other':
        return 'Other/Unknown'
    else:
        return race
    
score_label_df['race'] = score_label_df['race'].apply(race_simple)

score_label_df['mass_idx'] = score_label_df['lesion number']

def mal_label(mal):
    if mal == 'Malignant':
        return 1
    elif mal == 'Benign' or mal == 'HR Benign':
        return 0
    else:
        raise ValueError('Invalid malignancy')

score_label_df['malignancy_label'] = score_label_df['pathology'].apply(mal_label)

score_label_df.groupby(['mass_idx', 'malignancy_label', 'margin_label']).count()[['image_id']]

### Run Malignancy Prediction

In [ ]:
## Add Malignancy Predictions
predictions_wmal = ext.malignancy_for_models(score_label_df)

predictions_wmal.groupby('model_name').count()[['mal_pos_prob', 'mal_neg_prob', 'malignancy_pred']]

In [ ]:
__prob_sums = predictions_wmal[['mal_pos_prob', 'mal_neg_prob']].sum(axis=1).round(2).value_counts()
assert len(__prob_sums) == 1 and 1.0 in __prob_sums
__prob_sums

In [ ]:
for model in ext.models:
    predictions_wmal[predictions_wmal['model_name'] == model]['mal_pos_prob'].hist(alpha=0.5)

# Analysis
## ROC

### Summary Statistics

In [ ]:
summary = ext.patient_summary_statistics(predictions_wmal)[['subgroup', 'value', 'count']]
summary.to_csv(ext.external_validation_path/f'{ext.environment}_summary.csv', index=False)
summary

In [ ]:
ext.roc_figure_for_models(predictions_wmal)

# Export Key Results

## Margin ROC

In [ ]:
export_df = predictions_wmal.copy()

In [ ]:
margin_roc_df, margin_ci_df = ext.roc_dfs(export_df, 'margin', ext.margin_name_labels, 'margin_label')
margin_roc_df.head(5).pipe(display)
margin_ci_df.head(5).pipe(display)

In [ ]:
margin_roc_df.to_csv(ext.external_validation_path / f'{ext.environment}_margin_roc.csv', index=False)
margin_roc_df.groupby(['model', 'margin']).count().pipe(display)

margin_ci_df.to_csv(ext.external_validation_path / f'{ext.environment}_margin_ci.csv', index=False)
margin_ci_df[margin_ci_df['subgrouping'] == 'all'].pivot(index='margin', columns='model', values=['auc', 'ci_025', 'ci_975']).swaplevel(0,1,axis=1)[ext.models].round(2)

# Note the warnings come from the fact some subgroups only have one entry in the class. Obviously, that is not a valid test, but it's included for completeness.

## Activation Precision

In [ ]:
EXPORT_BOOTSTRAPS=100

In [ ]:
prototype_ap_df = ext.classwise_activation_precision_for_protopnets(num_bootstraps=EXPORT_BOOTSTRAPS)
prototype_ap_df

In [ ]:
gradcam_ap_df = ext.run_gradcams(num_bootstraps=EXPORT_BOOTSTRAPS)
gradcam_ap_df

In [ ]:
ap_ci_df = pd.concat([prototype_ap_df, gradcam_ap_df], sort=True)
ap_ci_df.to_csv(ext.external_validation_path / f'{ext.environment}_ap_ci.csv', index=False)
ap_ci_df.fillna('').groupby(['model', 'method']).mean().pipe(display)
ap_ci_df.pipe(display)

## Malignancy

In [ ]:
export_df[['mal_pos_prob', 'malignancy_label']].to_csv(ext.external_validation_path/f'{ext.environment}_mal_dist.csv', index=False)

In [ ]:
mal_roc_df, mal_ci_df = ext.roc_dfs(export_df, 'malignancy', ext.malignancy_name_labels, 'malignancy_label')
mal_roc_df.pipe(display)
mal_ci_df.pipe(display)

In [ ]:
mal_roc_df.to_csv(ext.external_validation_path / f'{ext.environment}_mal_roc.csv', index=False)
mal_roc_df.groupby(['model', 'malignancy']).count().pipe(display)

mal_ci_df.to_csv(ext.external_validation_path / f'{ext.environment}_mal_ci.csv', index=False)
mal_ci_df[mal_ci_df['subgrouping'] == 'all']\
    .pivot(index='malignancy', columns='model', values=['auc', 'ci_025', 'ci_975'])\
    .swaplevel(0,1,axis=1)[ext.models].round(2)

# Note the warnings come from the fact some subgroups only have one entry in the class. Obviously, that is not a valid test, but it's included for completeness.

In [ ]:
plt.figure()

# Plot ROC curve for each class
for model in ext.models:
    class_data = mal_roc_df[(mal_roc_df['model'] == model) & (mal_roc_df['malignancy'] == 'malignant')].sort_values('threshold')

    row = mal_ci_df[(mal_ci_df['subgrouping'] == 'all') & (mal_ci_df['model'] == model) & (mal_ci_df['malignancy'] == 'malignant')].round(2)

    auc_label = f'AUC: {row["auc"].item()} ({row["ci_025"].item()}, {row["ci_975"].item()})'

    plt.plot(class_data['fpr'], class_data['tpr'], label=f'Model {model}\n{auc_label}')

plt.plot([0, 1], [0, 1],'--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Malignancy')
plt.legend(loc='best')

# Show the plot
plt.show()